# RL is not enough when there are many right answers

**What reward optimization can and cannot do to a pretrained basis — a walkthrough with exact ground truth.**

This notebook is a self-contained, runnable version of the experiment in
[`experiments/rhm/rl_dimensionality/`](https://github.com/jagilley/abstraction/tree/main/experiments/rhm/rl_dimensionality)
of Jasper Gilley's research monorepo. It does three things:

1. **Explains the setup** — the Random Hierarchy Model as a language with a dial for "how many ways to say the same thing", the task, the two verifiers, and the exact Bayes references that make "how much did RL gain" a well-posed question.
2. **Runs a scaled-down replica live** (~15 min on a free Colab T4 GPU): pretrain to plateau at one value of the dial, then climb the attainment ladder — vanilla REINFORCE → KL-anchored REINFORCE → expert iteration — from the same checkpoint, with the same rollout budget, and read out both *task reward* and *what the model actually knows*.
3. **Shows the full-scale results** (5 dials × 3 mechanisms × 2 verifiers, embedded in the notebook) with the interpretation from the writeup inline.

> **Runtime → Change runtime type → GPU** before running. Everything runs on CPU too, just slowly; set `RUN_LIVE = False` in the config cell to skip the live training and only read the embedded results.

The substrate is the Random Hierarchy Model of Cagnetta & Wyart (*Phys. Rev. X*, 2024). Every number in the interpretation sections comes from the full-scale runs (single seed, one model size — see the scope note at the end).

## 1. The question

RL from a blank slate conquers games. On language we pretrain first and use reward only to finish. The obvious difference between the two domains is **how many right answers each state admits**: a game position has one best move; a prompt has many valid continuations that mean the same thing.

Two intuitions worth operationalizing in a setting where ground truth is exactly computable:

1. **RL degrades with semantic dimensionality** — RL-from-scratch works on semantically simple domains (games) but semantically complex domains (language) require pretraining, and RL's efficacy *relative to the domain* shrinks as the domain gets semantically richer.
2. **The representational-basis claim** — after a tweet asking *"does RL create new capabilities in the representational basis that was created by pretraining?"* If the answer is no, reward optimization can only rearrange, sharpen, or degrade what pretraining built — and models will get more and more fried even as they can operate on longer and more sophisticated tasks.

Neither claim is the stronger "RL can never create capability". The experiment is designed to *localize* where any capability change comes from, not to deny it.

Every result below is split into two categories:

- **Bounds (algorithm-free)** — exact Bayes ceilings and floors, computed by sum-product belief propagation on the known parse tree. No optimizer can beat these; no objection about RL variants can touch them.
- **Attainment (algorithm-specific)** — how much of the bounded headroom each training mechanism actually captures, and at what representational cost. To pre-empt "that was just your RL variant", attainment is measured for a *ladder* of three mechanisms ending in the class limit.

In [ ]:
#@title Setup { display-mode: "form" }
import os, math, json, time, gzip, base64, itertools
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
SMOKE = os.environ.get("RLDIM_SMOKE") == "1"   # tiny sizes for CI-style testing only
print(f"torch {torch.__version__} | device: {device}" + ("  [SMOKE]" if SMOKE else ""))

# --- palette (validated categorical + one sequential ramp) -------------------
PAL = dict(blue="#2a78d6", orange="#eb6834", aqua="#1baf7a", violet="#4a3aa7",
           red="#e34948", gray="#9a9891", light="#e6e4df", ink="#0b0b0b", muted="#52514e")
LEVEL_RAMP = ["#b7d3f6", "#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#0d366b"]  # L0..L5
MECH_COLOR = {"pretrained": PAL["gray"], "REINFORCE": PAL["violet"], "REINFORCE + KL": PAL["blue"],
              "expert iteration": PAL["orange"], "continued NTP": PAL["aqua"]}
VERIFIER_COLOR = {"exact": PAL["blue"], "parse": PAL["aqua"]}
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": PAL["light"], "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.edgecolor": PAL["muted"], "axes.labelcolor": PAL["ink"], "xtick.color": PAL["muted"],
    "ytick.color": PAL["muted"], "font.size": 10, "legend.frameon": False,
})
MS_FULL = [1, 2, 3, 4, 6]

In [ ]:
#@title Embedded full-scale results (5 m × {base, KL, EI} runs from the Modal volume) { display-mode: "form" }
FULL_RESULTS_B64 = """
...
"""
FULL = json.loads(gzip.decompress(base64.b64decode(FULL_RESULTS_B64.strip())))
del FULL_RESULTS_B64

def full_refs(m):      return FULL[str(m)]["refs"]
def full_pretrain(m):  return FULL[str(m)]["pretrain"]
def full_final(m, cond):
    ck = FULL[str(m)]["conditions"][cond]["checkpoints"]
    return ck[max(ck, key=int)]
def full_rounds(m, cond):
    ck = FULL[str(m)]["conditions"][cond]["checkpoints"]
    return [ck[k] for k in sorted(ck, key=int)]

print("full-scale runs loaded for m =", [int(k) for k in FULL],
      "| conditions:", sorted({c for k in FULL for c in FULL[k]["conditions"]}))

## 2. The Random Hierarchy Model, and a dial for "how many ways to say the same thing"

The **Random Hierarchy Model** (RHM) is a synthetic language with a known generative process. A sequence of $s^L$ tokens from a vocabulary of size $v$ is produced by a tree: a root feature is drawn uniformly, and every feature at every level expands into an $s$-tuple of lower-level features by picking one of its **$m$ interchangeable production rules** uniformly at random. After $L$ expansions the leaves are the tokens. The rules are sampled once and frozen — they *are* the grammar.

We use $v=8$, $s=2$, $L=6$: 64-token sequences, six levels of composition, ~2.7M-parameter transformer. Because the generating tree is known, we can compute *exactly* what any predictor could achieve, and *exactly* which levels of the hierarchy a trained model has and hasn't learned.

**The dial is $m$, synonymic multiplicity.** With **invertible rules** (every legal tuple maps back to exactly one (feature, rule) pair, so parses are exact and roots unique):

- $m=1$: each feature has exactly one way to say itself, so the whole language is exactly $v=8$ possible sequences — a deterministic surface, the **game** endpoint.
- $m=v^{s-1}=8$: every one of the $v^s=64$ possible tuples is a legal expansion of something, so the language is the uniform distribution over all $8^{64}$ strings — **noise**, no structure at all.
- Sequence entropy is $3 + 63\log_2 m$ bits out of a 192-bit maximum.

The sweep runs $m \in \{1, 2, 3, 4, 6\}$.

**Why $m$ and not depth $L$?** RHM semantic sample complexity is governed by $m^L$. Varying $L$ moves it — but through sequence length $s^L$, so horizon, context, and compute per rollout all change with it, and "RL degrades with horizon" (a boring, known result) would be indistinguishable from the semantic claim. Varying $m$ moves $m^L$ with the tree shape, sequence length, architecture, and rollout compute *exactly fixed*. And $m$ is literally the surface-to-latent degeneracy axis — how many surface forms each meaning has — which is the thing that separates games from language.

In [ ]:
# ----------------------------------------------------------------------------
# RHM data generation (ported from rhm_data.py)
# ----------------------------------------------------------------------------
def generate_rules_invertible(v, s, L, m, seed=0):
    """Rules that are COLLISION-FREE across features at every level: the v*m produced
    s-tuples are all distinct, so every legal tuple maps to exactly one (feature, rule).
    rules[ell] has shape (v, m, s); level 0 is the root's rules, level L-1 produces tokens."""
    rng = np.random.default_rng(seed)
    n_codes = v ** s
    assert v * m <= n_codes, "collision-free rules require v*m <= v**s"
    rules = []
    for _ in range(L):
        codes = rng.choice(n_codes, size=v * m, replace=False).reshape(v, m)
        layer = np.empty((v, m, s), dtype=np.int64)
        for i in range(s):
            layer[:, :, i] = (codes // (v ** i)) % v
        rules.append(layer)
    return rules

def generate_sequences_batched(rules, n_sequences, seed=0, batch_size=10000):
    """Sample sequences: uniform root, then uniform rule choice at every internal node."""
    rng = np.random.default_rng(seed)
    L = len(rules); v, m, s = rules[0].shape
    out = np.empty((n_sequences, s ** L), dtype=np.int64)
    for start in range(0, n_sequences, batch_size):
        bs = min(batch_size, n_sequences - start)
        cur = rng.integers(0, v, size=(bs, 1))
        for ell in range(L):
            rc = rng.integers(0, m, size=cur.shape)
            nxt = np.empty((bs, cur.shape[1] * s), dtype=np.int64)
            for j in range(cur.shape[1]):
                nxt[:, j * s:(j + 1) * s] = rules[ell][cur[:, j], rc[:, j]]
            cur = nxt
        out[start:start + bs] = cur
    return out

def generate_with_traces(rules, n_sequences, seed=999):
    """Like generate_sequences_batched but also returns the latent tree:
    level_features[ell] (n, s^ell) = features at depth ell (0 = root), level_rules[ell] = rule choices."""
    rng = np.random.default_rng(seed)
    L = len(rules); v, m, s = rules[0].shape
    level_features, level_rules = [], []
    cur = rng.integers(0, v, size=(n_sequences, 1))
    for ell in range(L):
        level_features.append(cur.copy())
        rc = rng.integers(0, m, size=cur.shape)
        level_rules.append(rc.copy())
        nxt = np.empty((n_sequences, cur.shape[1] * s), dtype=np.int64)
        for j in range(cur.shape[1]):
            nxt[:, j * s:(j + 1) * s] = rules[ell][cur[:, j], rc[:, j]]
        cur = nxt
    level_features.append(cur.copy())
    return cur, level_features, level_rules

def position_levels(seq_len, s):
    """Hierarchy level of each *prediction* position 1..seq_len-1 = s-adic valuation of the position.
    Predicting position p means crossing a level-k boundary where s^k | p — the deeper the level,
    the more of the tree the predictor must have resolved."""
    levels = np.zeros(seq_len - 1, dtype=np.int64)
    for t in range(seq_len - 1):
        p, level = t + 1, 0
        while p % s == 0:
            p //= s; level += 1
        levels[t] = level
    return levels

def suffix_position_levels(prefix_len, seq_len, s):
    levels = []
    for i in range(prefix_len, seq_len):
        p, level = i, 0
        while p > 0 and p % s == 0:
            p //= s; level += 1
        levels.append(level)
    return levels

V, S, DEPTH = 8, 2, 6
SEQ_LEN = S ** DEPTH; PREFIX_LEN = SEQ_LEN // 2; SUFFIX_LEN = SEQ_LEN - PREFIX_LEN
LETTERS = "abcdefgh"
tostr = lambda seq: "".join(LETTERS[t] for t in seq)

print("Sequence entropy, bits (of 192 max):",
      {m: round(3 + 63 * math.log2(m), 1) for m in [1, 2, 3, 4, 6, 8]})
for m in [1, 4]:
    rules = generate_rules_invertible(V, S, DEPTH, m, seed=0)
    seqs, feats, _ = generate_with_traces(rules, 3000, seed=1)
    roots = feats[0][:, 0]
    print(f"\nm={m}: {len({tostr(x) for x in seqs})} distinct sequences among 3000 samples")
    for root in [0, 1]:
        ex = seqs[roots == root][:3]
        print(f"  root {root}:")
        for x in ex:
            print("    ", tostr(x))

At $m=1$ every root has exactly one surface form — there are only 8 sentences in the language. At $m=4$ the same root produces sequences that share no tokens at all, yet parse to the same meaning. That is the synonymy the dial controls.

Here is one sequence's parse tree — the latent structure a model would have to recover to predict deep positions:

In [ ]:
def plot_parse_tree(rules, m, seed=7):
    seq, feats, rc = generate_with_traces(rules, 1, seed=seed)
    L = len(rules)
    fig, ax = plt.subplots(figsize=(13, 3.6))
    for ell in range(L + 1):
        row = feats[ell][0]; n = len(row); w = SEQ_LEN / n
        y = L - ell
        for j, f in enumerate(row):
            color = LEVEL_RAMP[5 - ell] if ell < L else PAL["light"]      # root darkest, tokens light
            ax.add_patch(Rectangle((j * w + 0.06, y - 0.42), w - 0.12, 0.84, facecolor=color, edgecolor="white", linewidth=0.8))
            if ell < L or n <= 64:
                ax.text(j * w + w / 2, y, LETTERS[f], ha="center", va="center",
                        color="white" if ell <= 2 else PAL["ink"], fontsize=9 if ell < L else 7, fontweight="bold")
    ax.set_xlim(0, SEQ_LEN); ax.set_ylim(-0.6, L + 0.6); ax.grid(False)
    ax.set_yticks(range(L + 1)); ax.set_yticklabels(["tokens (64)"] + [f"L{k} nodes ({2 ** (L - k)})" for k in range(1, L)] + ["root"])
    ax.set_xticks([]); ax.spines["bottom"].set_visible(False); ax.spines["left"].set_visible(False)
    ax.set_title(f"One RHM sequence and its generating tree (m={m}): each node picked 1 of {m} rules", loc="left")
    plt.tight_layout(); plt.show()

plot_parse_tree(generate_rules_invertible(V, S, DEPTH, 4, seed=0), m=4)

## 3. The task, and two ways to grade it

**Task:** the prefix is the first 32 tokens of a 64-token sequence; the policy generates the remaining 32 autoregressively (temperature-1 sampling during training; greedy *and* sampled at eval).

**Two verifiers**, both run for every configuration:

- **exact** — fraction of suffix tokens matching *the one sampled ground-truth suffix*. A **canonical-answer** verifier: a synonym counts as *wrong*. The synonym lottery is part of the reward.
- **parse** — bottom-up parse of prefix + generation with the known grammar; reward = mean over hierarchy levels of the fraction of nodes that are valid (invalid nodes propagate empty possible-sets upward). A **validity** verifier: synonymy-invariant (any grammatical continuation is accepted), graded in composition depth, dense at the bottom levels. Closer to RLHF/RLVR, where raters don't penalize paraphrase.

The parse verifier's root-level component — "does the whole 64-token string parse all the way to a root?" — is the metric that *requires* deep structure.

In [ ]:
def parse_valid_fractions_torch(seqs, rules_t):
    """Per-level fraction of valid nodes in the bottom-up possible-set (CYK-style) parse.
    seqs: (B, s^L) long. rules_t: list of L (v, m, s) long tensors. Returns (B, L):
    column k = fraction of nodes with a non-empty possible-set after fold k+1 (k=L-1 is the root)."""
    B = seqs.shape[0]; v, m, s = rules_t[0].shape; L = len(rules_t)
    cur = F.one_hot(seqs, v).bool()                                  # (B, n, v) possible-sets at the leaves
    fracs = []
    for ell in range(L - 1, -1, -1):
        n2 = cur.shape[1] // s
        children = cur.view(B, n2, s, v)
        ok = torch.ones(B, n2, v, m, dtype=torch.bool, device=seqs.device)
        for i in range(s):
            idx = rules_t[ell][:, :, i].reshape(-1)                  # (v*m,) child-i feature of each (parent, rule)
            ok &= children[:, :, i, :][:, :, idx].view(B, n2, v, m)
        cur = ok.any(dim=3)                                          # (B, n2, v) parent possible-sets
        fracs.append(cur.any(dim=2).float().mean(dim=1))
    return torch.stack(fracs, dim=1)

def reward_fn(prefix, generated, true_suffix, rw, rules_t):
    if rw == "exact":
        return (generated == true_suffix).float().mean(dim=1)
    full = torch.cat([prefix, generated], dim=1)
    return parse_valid_fractions_torch(full, rules_t).mean(dim=1)

# Sanity check on m=4: on-grammar sequences parse fully; random suffixes fail high up; a corrupted tail fails partially
rules = generate_rules_invertible(V, S, DEPTH, 4, seed=0)
rules_t = [torch.from_numpy(r).long().to(device) for r in rules]
on = torch.from_numpy(generate_sequences_batched(rules, 200, seed=1)).to(device)
rng = np.random.default_rng(0)
rand = on.clone(); rand[:, PREFIX_LEN:] = torch.from_numpy(rng.integers(0, V, size=(200, SUFFIX_LEN))).to(device)
corrupt = on.clone(); corrupt[:, -3:] = torch.from_numpy(rng.integers(0, V, size=(200, 3))).to(device)
for name, x in [("on-grammar", on), ("random suffix", rand), ("last 3 tokens corrupted", corrupt)]:
    pf = parse_valid_fractions_torch(x, rules_t).mean(dim=0).cpu().numpy()
    print(f"{name:>24s}: parse reward={pf.mean():.3f}  per level L1..L6(root)=[{' '.join(f'{p:.2f}' for p in pf)}]")

## 4. Part 1 — Bounds (algorithm-free)

"How much did RL gain?" is only well-posed relative to what is achievable. Because the generating tree and its rules are known, **sum-product belief propagation** on the parse tree gives exact posteriors: one upward pass (leaves → root) and one downward pass (root → leaves) with the prefix observed and the suffix free yields $P(x_i = a \mid \text{prefix})$ for every suffix position. From these:

- **Exact-match greedy ceiling** $= \mathbb{E}\big[\tfrac{1}{32}\sum_i \max_a P(x_i=a\mid\text{prefix})\big]$ — the best *any* policy can do on the exact verifier.
- **Sampled-policy reference** $= \mathbb{E}\big[\tfrac{1}{32}\sum_i \sum_a P(x_i=a\mid\text{prefix})^2\big]$ — what a perfectly *calibrated* sampling policy scores. The gap between the two is pure policy sharpening, requiring no new knowledge.
- **Prefix-blind floor** $= \tfrac{1}{32}\sum_i \max_a P(x_i=a)$ — the best predictor that ignores the prefix entirely (position marginals); **uniform floor** $= 1/v$.
- **NTP Bayes floor** $= H(x_i \mid x_{<i})$ per position (the same BP with $x_{<i}$ observed), grouped by hierarchy level — the irreducible next-token loss. A model's loss *above* this floor, per level, is exactly how much of that level's structure it has not learned.
- **Parse floors** by Monte Carlo (uniform-random suffix; blind-argmax suffix); ceiling $=1$.

The BP marginals were verified against brute-force enumeration on small trees (max error ~1e-16). The cell below recomputes the references live for every $m$.

In [ ]:
# ----------------------------------------------------------------------------
# Exact references via sum-product BP on the parse tree (ported from rhm_bayes_entropy.py / rl_dim_ablation.py)
# ----------------------------------------------------------------------------
def _upward(up_leaf, rules, B, s, L, v):
    up = [None] * (L + 1); up[L] = up_leaf
    for d in range(L - 1, -1, -1):
        n_par = s ** d
        up_child = up[d + 1].reshape(B, n_par, s, v)
        up_d = np.zeros((B, n_par, v)); rd = rules[d]
        for a in range(v):
            acc = np.zeros((B, n_par))
            for r in range(rd.shape[1]):
                tup = rd[a, r]; prod = np.ones((B, n_par))
                for k in range(s):
                    prod = prod * up_child[:, :, k, tup[k]]
                acc += prod
            up_d[:, :, a] = acc / rd.shape[1]
        up[d] = up_d
    return up

def _downward(up, rules, B, s, L, v):
    down = [None] * (L + 1); down[0] = np.full((B, 1, v), 1.0 / v)
    for d in range(L):
        n_par = s ** d
        up_child = up[d + 1].reshape(B, n_par, s, v)
        rd = rules[d]; m = rd.shape[1]
        down_child = np.zeros((B, n_par, s, v))
        for a in range(v):
            da = down[d][:, :, a]
            for r in range(m):
                tup = rd[a, r]
                ups = [up_child[:, :, k, tup[k]] for k in range(s)]
                for i in range(s):
                    prod = np.ones((B, n_par))
                    for k in range(s):
                        if k != i:
                            prod = prod * ups[k]
                    down_child[:, :, i, tup[i]] += da * (prod / m)
        down[d + 1] = down_child.reshape(B, n_par * s, v)
    return down

def _posterior(rules, up_leaf):
    v, m, s = rules[0].shape; L = len(rules); B = up_leaf.shape[0]
    up = _upward(up_leaf, rules, B, s, L, v); down = _downward(up, rules, B, s, L, v)
    post = up[L] * down[L]
    return post / np.clip(post.sum(axis=2, keepdims=True), 1e-30, None)

def suffix_marginals(rules, seqs, prefix_len):
    """Exact P(x_i = a | x_<prefix_len) for every suffix position: (B, suffix_len, v)."""
    v = rules[0].shape[0]; seq_len = seqs.shape[1]; B = seqs.shape[0]
    up_leaf = np.ones((B, seq_len, v))
    oh = np.zeros((B, prefix_len, v)); np.put_along_axis(oh, seqs[:, :prefix_len, None], 1.0, axis=2)
    up_leaf[:, :prefix_len] = oh
    return _posterior(rules, up_leaf)[:, prefix_len:]

def blind_marginals(rules):
    v, m, s = rules[0].shape; L = len(rules)
    return _posterior(rules, np.ones((1, s ** L, v)))[0]          # (seq_len, v)

def bp_conditional_entropies(rules, seqs):
    """Exact H(x_i | x_<i) in nats for every position, averaged over seqs."""
    B, seq_len = seqs.shape; v = rules[0].shape[0]
    H = np.zeros(seq_len)
    for i in range(seq_len):
        up_leaf = np.ones((B, seq_len, v))
        if i > 0:
            oh = np.zeros((B, i, v)); np.put_along_axis(oh, seqs[:, :i, None], 1.0, axis=2); up_leaf[:, :i] = oh
        post = _posterior(rules, up_leaf)[:, i]
        H[i] = (-(post * np.log(np.clip(post, 1e-30, None))).sum(1)).mean()
    return H

def compute_references(rules, n_em=300, n_ntp=100, seed=0):
    v, m, s = rules[0].shape; L = len(rules); seq_len = s ** L; p = seq_len // 2
    seqs = generate_sequences_batched(rules, max(n_em, n_ntp), seed=seed)
    levels = np.array(suffix_position_levels(p, seq_len, s))
    post = suffix_marginals(rules, seqs[:n_em], p)
    blind = blind_marginals(rules)[p:]
    H = bp_conditional_entropies(rules, seqs[:n_ntp])
    pl = position_levels(seq_len, s)
    per_level_floor = [float(H[1:][pl == l].mean()) for l in range(L)]
    # parse floors by Monte Carlo
    rules_t_ = [torch.from_numpy(r).long() for r in rules]
    pre = torch.from_numpy(seqs[:n_em, :p])
    rs = torch.from_numpy(np.random.default_rng(seed + 1).integers(0, v, size=(n_em, seq_len - p)))
    bs = torch.from_numpy(np.tile(blind.argmax(1), (n_em, 1)))
    pf_rand = parse_valid_fractions_torch(torch.cat([pre, rs], 1), rules_t_).mean(0)
    pf_blind = parse_valid_fractions_torch(torch.cat([pre, bs], 1), rules_t_).mean(0)
    return {
        "m": m, "uniform_nats": float(np.log(v)),
        "em_greedy_ceiling": float(post.max(2).mean()), "em_sampled_ref": float((post ** 2).sum(2).mean()),
        "em_blind_floor": float(blind.max(1).mean()), "em_uniform": 1.0 / v,
        "em_greedy_ceiling_per_level": [float(post.max(2)[:, levels == l].mean()) for l in range(L)],
        "ntp_floor_pos1plus": float(H[1:].mean()), "ntp_floor_per_level": per_level_floor,
        "parse_random_floor": float(pf_rand.mean()), "parse_blind_floor": float(pf_blind.mean()),
        "parse_blind_floor_per_level": pf_blind.tolist(),
    }

REFS = {}
t0 = time.time()
for m in MS_FULL:
    REFS[m] = compute_references(generate_rules_invertible(V, S, DEPTH, m, seed=0),
                                 n_em=60 if SMOKE else 300, n_ntp=20 if SMOKE else 100)
print(f"computed in {time.time() - t0:.0f}s\n")
print(f"{'m':>2} | {'EM greedy ceil':>14} | {'EM sampled ref':>14} | {'EM blind floor':>14} | {'headroom':>8} | {'parse blind floor':>17} | {'NTP Bayes floor':>15}")
for m, r in REFS.items():
    print(f"{m:>2} | {r['em_greedy_ceiling']:>14.4f} | {r['em_sampled_ref']:>14.4f} | {r['em_blind_floor']:>14.4f} | "
          f"{r['em_greedy_ceiling'] - r['em_blind_floor']:>8.4f} | {r['parse_blind_floor']:>17.4f} | {r['ntp_floor_pos1plus']:>15.4f} nats")
print(f"\n(uniform NTP floor = ln 8 = {np.log(8):.4f} nats; exact-match uniform floor = 1/8; parse ceiling = 1.0 at every m)")

In [ ]:
def plot_bounds(refs_by_m, title_suffix=""):
    ms = sorted(refs_by_m)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1.5, 1]})
    x = np.arange(len(ms))
    for i, m in enumerate(ms):
        r = refs_by_m[m]
        a1.bar(i, r["em_blind_floor"] - 1 / 8, bottom=1 / 8, width=0.62, color=PAL["gray"])
        a1.bar(i, r["em_greedy_ceiling"] - r["em_blind_floor"], bottom=r["em_blind_floor"], width=0.62, color=PAL["orange"])
        a1.text(i, r["em_greedy_ceiling"] + 0.015, f"{r['em_greedy_ceiling'] - r['em_blind_floor']:.4f}", ha="center", fontsize=9, color=PAL["ink"])
    a1.axhline(1 / 8, color=PAL["muted"], lw=0.8, ls="--"); a1.text(len(ms) - 0.55, 0.13, "uniform guess 1/8", fontsize=8, color=PAL["muted"], ha="right")
    a1.set_xticks(x); a1.set_xticklabels([f"m={m}" for m in ms]); a1.set_ylim(0, 1)
    a1.set_ylabel("exact-match accuracy"); a1.set_title("Exact-match verifier: what is achievable at all" + title_suffix, loc="left")
    a1.bar([], [], color=PAL["gray"], label="position marginals (uniform → blind floor): no structure needed")
    a1.bar([], [], color=PAL["orange"], label="prefix-conditional headroom (blind floor → Bayes ceiling): needs understanding")
    a1.legend(loc="upper right", fontsize=8)
    a2.bar(x, [refs_by_m[m]["parse_blind_floor"] for m in ms], width=0.62, color=PAL["gray"], label="blind floor")
    a2.bar(x, [1 - refs_by_m[m]["parse_blind_floor"] for m in ms], bottom=[refs_by_m[m]["parse_blind_floor"] for m in ms], width=0.62, color=PAL["aqua"], label="headroom to ceiling (1.0)")
    a2.set_xticks(x); a2.set_xticklabels([f"m={m}" for m in ms]); a2.set_ylim(0, 1.05)
    a2.set_ylabel("parse reward"); a2.set_title("Parse verifier", loc="left"); a2.legend(loc="lower right", fontsize=8)
    plt.tight_layout(); plt.show()

plot_bounds(REFS, " (computed live)")

### What the bounds say

Two structural facts, both pure computation (the full-scale numbers, with 1000 prefixes: greedy ceiling − blind floor = **0.096** at $m=1$, **0.005** at $m=2$, **~0.0005** at $m=4$, **~0.0000** at $m=6$):

**B1. The canonical-answer verifier's semantic content vanishes with $m$.** Deep hierarchies mix: prefix→suffix correlations cross the root and are per-token negligible at $L=6$. Beyond $m=1$, essentially everything the exact verifier can reward decomposes into **position-marginal learning** (uniform → blind floor, which needs no structure) and **policy sharpening** (sampled reference → greedy ceiling, e.g. 0.191 → 0.279 at $m=2$, which needs no *new* knowledge). No RL variant can be exempted from this: there is nothing to optimize toward. (Corollary for an earlier line of work in the repo: an RL generation gain of 27.8% → 38.6% under sampled eval is consistent with sharpening toward the greedy ceiling rather than new structure.)

**B2. The two verifiers order "dimensionality" in opposite directions.** Grammar occupancy $m/v$ rises with $m$, so a *validity* verifier's reward gets **denser** as synonymy grows (more strings are grammatical) while a *canonical-answer* verifier's gets emptier. "Semantic dimensionality" is not one axis for RL; it is a property of the **verifier × domain pair**.

Also structural: even at $m=1$ the exact-match ceiling is 0.842, not 1.0 — distinct root rules can share the same *left* child, so the prefix doesn't always determine the root. The BP references, not intuition, define what's achievable at every $m$.

## 5. The model, and pretraining to plateau

A minimal GPT-2 (6 layers, 6 heads, 192-d, ~2.7M parameters at full scale) trained on **sequence-aligned** next-token prediction — each row is one full 64-token sequence — so per-position validation loss is exactly comparable to the BP Bayes floor and position embeddings are hierarchy-aligned.

**Pretrain to plateau, not to budget.** Each $m$ pretrains until aligned val loss stops improving (best checkpoint kept). The claims concern what reward optimization does to a *given* basis; fixed-budget pretraining would confound "RL got dumber" with "pretraining got worse".

Three **representational readouts** are taken at every checkpoint of every condition:

- **NTP val loss vs the Bayes floor** — excess = distributional damage — and **per-level NTP excess**: which hierarchy levels the model actually knows. A level at excess ≈ (ln 8 − floor) is unlearned; at ≈ 0 it is fully learned.
- **Per-layer feature $\eta^2$** against the ground-truth hierarchy: how much of the variance in layer activations (at the last position) is explained by the identity of the sequence's ancestor at each level — whether the layers carry hierarchical structure at all.
- **Per-level generation accuracy**, and the **greedy-vs-sampled gap** (policy entropy).

In [ ]:
# ----------------------------------------------------------------------------
# Minimal GPT (ported from model.py) with intermediate-activation readout
# ----------------------------------------------------------------------------
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        self.n_head, self.head_dim = n_head, n_embd // n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd); self.c_proj = nn.Linear(n_embd, n_embd)
        self.register_buffer("bias", torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size))
    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(C, dim=2)
        q, k, v = [t.view(B, T, self.n_head, self.head_dim).transpose(1, 2) for t in (q, k, v)]
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))
        y = F.softmax(att, dim=-1) @ v
        return self.c_proj(y.transpose(1, 2).contiguous().view(B, T, C))

class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd); self.attn = CausalSelfAttention(n_embd, n_head, block_size)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd))
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        return x + self.mlp(self.ln_2(x))

class GPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_layer, n_head, n_embd):
        super().__init__()
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(vocab_size, n_embd), wpe=nn.Embedding(block_size, n_embd),
            h=nn.ModuleList([Block(n_embd, n_head, block_size) for _ in range(n_layer)]), ln_f=nn.LayerNorm(n_embd)))
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.apply(self._init)
    def _init(self, mod):
        if isinstance(mod, (nn.Linear, nn.Embedding)):
            nn.init.normal_(mod.weight, mean=0.0, std=0.02)
            if isinstance(mod, nn.Linear) and mod.bias is not None: nn.init.zeros_(mod.bias)
    def forward(self, idx, targets=None, return_intermediates=False):
        B, T = idx.size()
        x = self.transformer.wte(idx) + self.transformer.wpe(torch.arange(T, device=idx.device))
        inter = {}
        for i, block in enumerate(self.transformer.h):
            x = block(x)
            if return_intermediates: inter[f"post_block{i}"] = x
        logits = self.lm_head(self.transformer.ln_f(x))
        loss = None if targets is None else F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return (logits, loss, inter) if return_intermediates else (logits, loss)

def generate_suffix(model, prefix, suffix_len, temperature=1.0, greedy=False):
    x = prefix; outs = []
    for _ in range(suffix_len):
        logits, _ = model(x); nl = logits[:, -1, :]
        nt = nl.argmax(-1, keepdim=True) if greedy else torch.multinomial(F.softmax(nl / temperature, -1), 1)
        outs.append(nt); x = torch.cat([x, nt], dim=1)
    return torch.cat(outs, dim=1)

In [ ]:
#@title Live-replica configuration (scaled down from the full run; edit freely) { display-mode: "form" }
RUN_LIVE = True         #@param {type:"boolean"}
M_DEMO = 4              #@param [1, 2, 3, 4, 6] {type:"raw"}
VERIFIER = "parse"      #@param ["parse", "exact"]
SEED = 42

# model (full run: 6L/6H/192D) and budgets (full run: pretrain min 3K/max 30K, patience 8x250; 6000 RL steps x 64 = 384K rollouts; EI 6 x 4000 x 16)
N_LAYER, N_HEAD, N_EMBD = 6, 6, 192
PRE_MIN, PRE_MAX, PRE_EVAL_EVERY, PATIENCE, MIN_DELTA = 2000, 8000, 250, 4, 0.003
RL_STEPS, CKPT_EVERY, BATCH, LR, KL_COEF = 1200, 400, 64, 3e-4, 0.1
EI_ROUNDS, EI_PROMPTS, EI_N, EI_SFT_EPOCHS = 4, 1200, 16, 3          # 4 x 1200 x 16 = 76.8K rollouts = RL_STEPS x BATCH
N_TRAIN, N_VAL, N_EVAL, N_GEN_EVAL = 50000, 1000, 2000, 500
if SMOKE:
    N_LAYER, N_HEAD, N_EMBD = 2, 2, 64
    PRE_MIN, PRE_MAX, PRE_EVAL_EVERY, PATIENCE = 40, 80, 20, 2
    RL_STEPS, CKPT_EVERY = 12, 6
    EI_ROUNDS, EI_PROMPTS, EI_N, EI_SFT_EPOCHS = 2, 48, 4, 1
    N_TRAIN, N_VAL, N_EVAL, N_GEN_EVAL = 2000, 128, 128, 64
print(f"live replica: m={M_DEMO}, verifier={VERIFIER}, model {N_LAYER}L/{N_HEAD}H/{N_EMBD}D, "
      f"RL budget {RL_STEPS * BATCH:,} rollouts, EI budget {EI_ROUNDS * EI_PROMPTS * EI_N:,} rollouts")

In [ ]:
# ----------------------------------------------------------------------------
# Data for the live replica + evaluation machinery (ported from rl_dim_ablation.py)
# ----------------------------------------------------------------------------
rules = generate_rules_invertible(V, S, DEPTH, M_DEMO, seed=0)
rules_t = [torch.from_numpy(r).long().to(device) for r in rules]
refs = REFS[M_DEMO]
train_t = torch.from_numpy(generate_sequences_batched(rules, N_TRAIN, seed=SEED + 1))
val_t = torch.from_numpy(generate_sequences_batched(rules, N_VAL, seed=SEED + 2)).to(device)
rl_t = torch.from_numpy(generate_sequences_batched(rules, N_TRAIN, seed=SEED + 3))
eval_seqs, level_features, level_rules = generate_with_traces(rules, N_EVAL, seed=12345)
eval_t = torch.from_numpy(eval_seqs).to(device)
gen_eval_t = torch.from_numpy(generate_sequences_batched(rules, N_GEN_EVAL, seed=SEED + 4)).to(device)
SUFFIX_LEVELS = torch.tensor(suffix_position_levels(PREFIX_LEN, SEQ_LEN, S))
POS_LEVELS = position_levels(SEQ_LEN, S)
UNIFORM = float(np.log(V))

train_gen, rl_gen, ei_gen = (torch.Generator().manual_seed(SEED + k) for k in (0, 100, 200))
def ntp_batch():
    seqs = train_t[torch.randint(N_TRAIN, (BATCH,), generator=train_gen)].to(device)
    return seqs[:, :-1].contiguous(), seqs[:, 1:].contiguous()
def rl_batch():
    seqs = rl_t[torch.randint(N_TRAIN, (BATCH,), generator=rl_gen)].to(device)
    return seqs[:, :PREFIX_LEN], seqs[:, PREFIX_LEN:]
def make_gpt(): return GPT(V, SEQ_LEN, N_LAYER, N_HEAD, N_EMBD).to(device)

@torch.no_grad()
def eval_val_loss(model, n_batches=None):
    model.eval(); losses = []
    for i in range(0, len(val_t), BATCH):
        b = val_t[i:i + BATCH]; losses.append(model(b[:, :-1].contiguous(), b[:, 1:].contiguous())[1].item())
        if n_batches and len(losses) >= n_batches: break
    return float(np.mean(losses))

@torch.no_grad()
def eval_per_level_ntp(model):
    """Per-level cross-entropy on aligned val sequences (targets = positions 1+)."""
    model.eval(); per_pos = np.zeros(SEQ_LEN - 1); n = 0
    for i in range(0, len(val_t), BATCH):
        b = val_t[i:i + BATCH]
        lp = F.log_softmax(model(b[:, :-1].contiguous())[0], dim=-1)
        per_pos += (-lp.gather(2, b[:, 1:].unsqueeze(-1)).squeeze(-1)).sum(0).cpu().numpy(); n += b.shape[0]
    per_pos /= n
    out = {f"L{l}": float(per_pos[POS_LEVELS == l].mean()) for l in range(DEPTH)}
    out["overall"] = float(per_pos.mean()); return out

@torch.no_grad()
def eval_generation(model):
    model.eval(); out = {}
    for mode in ("greedy", "sampled"):
        ems, pfs = [], []
        for i in range(0, N_GEN_EVAL, BATCH):
            b = gen_eval_t[i:i + BATCH]; prefix, true = b[:, :PREFIX_LEN], b[:, PREFIX_LEN:]
            gen = generate_suffix(model, prefix, SUFFIX_LEN, greedy=(mode == "greedy"))
            ems.append((gen == true).float().cpu()); pfs.append(parse_valid_fractions_torch(torch.cat([prefix, gen], 1), rules_t).cpu())
        em, pf = torch.cat(ems), torch.cat(pfs)
        out[mode] = {"exact": float(em.mean()), "parse": float(pf.mean()), "parse_per_level": pf.mean(0).tolist(),
                     "exact_per_level": [float(em[:, SUFFIX_LEVELS == l].mean()) for l in range(DEPTH)]}
    return out

def hierarchy_eta2(acts, level_features, level_rules):
    """Fraction of last-position activation variance explained by the ancestor feature at each level."""
    last = acts[:, -1, :]; mean = last.mean(0); ss_tot = float(((last - mean) ** 2).sum())
    if ss_tot < 1e-12: return [0.0] * DEPTH
    def between(labels):
        ss = 0.0
        for val in np.unique(labels):
            mask = labels == val; ss += mask.sum() * float(((last[mask].mean(0) - mean) ** 2).sum())
        return ss / ss_tot
    n = acts.shape[0]
    return [between(level_features[l][:n, (SEQ_LEN - 1) // (S ** (DEPTH - l))]) for l in range(DEPTH)]

@torch.no_grad()
def eval_eta2(model):
    model.eval(); acts = {f"post_block{i}": [] for i in range(N_LAYER)}
    for i in range(0, len(eval_t), BATCH):
        _, _, inter = model(eval_t[i:i + BATCH], return_intermediates=True)
        for k in acts: acts[k].append(inter[k].cpu().numpy())
    return {k: hierarchy_eta2(np.concatenate(v), level_features, level_rules) for k, v in acts.items()}

def full_eval(model, label=""):
    ck = {"val_loss": eval_val_loss(model), "per_level_ntp": eval_per_level_ntp(model),
          "generation": eval_generation(model), "feature_eta2": eval_eta2(model)}
    g, s_ = ck["generation"]["greedy"], ck["generation"]["sampled"]
    print(f"  [{label:>22s}] NTP excess={ck['val_loss'] - refs['ntp_floor_pos1plus']:+.3f} nats | "
          f"greedy: exact={g['exact']:.3f} parse={g['parse']:.3f} | sampled: exact={s_['exact']:.3f} parse={s_['parse']:.3f} root={s_['parse_per_level'][-1]:.3f}")
    return ck

def excess_per_level(ck, floors): return [ck["per_level_ntp"][f"L{l}"] - floors[l] for l in range(DEPTH)]
def captured_per_level(ck, floors, uniform):
    """Share of each level's learnable structure the model has learned; None where the floor is at chance (nothing to predict)."""
    return [None if uniform - floors[l] < 0.05 else (uniform - ck["per_level_ntp"][f"L{l}"]) / (uniform - floors[l]) for l in range(DEPTH)]

def plot_basis(axes_or_ax, cks, floors, uniform, titles, suptitle=None):
    """Horizontal 'basis' bars: per level, share of learnable structure learned. Red dashed line = basis boundary."""
    axes = np.atleast_1d(axes_or_ax)
    for ax, ck, title in zip(axes, cks, titles):
        cap = captured_per_level(ck, floors, uniform)
        for l in range(DEPTH):
            if cap[l] is None:
                ax.add_patch(Rectangle((0, l - 0.38), 1, 0.76, facecolor="none", edgecolor=PAL["gray"], hatch="////", lw=0))
            else:
                ax.barh(l, max(cap[l], 0), color=LEVEL_RAMP[l], height=0.76)
                ax.text(max(cap[l], 0) + 0.02, l, f"{cap[l]:.2f}", va="center", fontsize=8, color=PAL["ink"])
        boundary = 0
        while boundary < DEPTH and cap[boundary] is not None and cap[boundary] >= 0.5: boundary += 1
        ax.axhline(boundary - 0.5, color=PAL["red"], ls="--", lw=1.4)
        ax.set_xlim(0, 1.25); ax.set_ylim(-0.6, DEPTH - 0.4); ax.set_yticks(range(DEPTH)); ax.set_yticklabels([f"L{l}" for l in range(DEPTH)])
        ax.set_xticks([0, 0.5, 1]); ax.set_title(title, loc="left", fontsize=10); ax.grid(axis="y", visible=False)
    axes[0].set_ylabel("hierarchy level (L0 = local, L5 = root)")
    if suptitle: axes[0].figure.suptitle(suptitle, x=0.01, ha="left", fontsize=11)

### Pretrain to plateau (live)

Watch the per-level loss: the low levels (local structure) are learned first and fully; at $m \ge 4$ the deep levels stay near chance no matter how long we train this model. That is the **basis boundary** — the level above which the model knows nothing — and everything that follows starts from it.

In [ ]:
def pretrain_to_plateau():
    torch.manual_seed(SEED)
    model = make_gpt(); opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    best_val, best_state, stale, curve, step = float("inf"), None, 0, [], 0
    t0 = time.time()
    while step < PRE_MAX:
        model.train(); x, y = ntp_batch(); _, loss = model(x, y)
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); step += 1
        if step % PRE_EVAL_EVERY == 0:
            vl = eval_val_loss(model); curve.append((step, vl))
            if vl < best_val - MIN_DELTA:
                best_val, stale = vl, 0; best_state = {k: p.detach().cpu().clone() for k, p in model.state_dict().items()}
            else: stale += 1
            if step % (PRE_EVAL_EVERY * 4) == 0: print(f"  step {step:5d}: val={vl:.4f} (best {best_val:.4f}, floor {refs['ntp_floor_pos1plus']:.4f}, stale {stale})  [{time.time() - t0:.0f}s]")
            if step >= PRE_MIN and stale >= PATIENCE: print(f"  plateau at step {step}"); break
    if best_state is None: best_state = {k: p.detach().cpu().clone() for k, p in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, best_state, best_val, step, curve

if RUN_LIVE:
    torch.manual_seed(SEED); init_state = {k: p.detach().cpu().clone() for k, p in make_gpt().state_dict().items()}
    print(f"Pretraining at m={M_DEMO} (Bayes floor {refs['ntp_floor_pos1plus']:.4f} nats, uniform {UNIFORM:.4f})")
    model, pretrained_state, pre_val, pre_steps, pre_curve = pretrain_to_plateau()
    captured = (UNIFORM - pre_val) / (UNIFORM - refs["ntp_floor_pos1plus"])
    print(f"\nPretrained: {pre_steps} steps, val={pre_val:.4f}, captured excess entropy={captured:.3f}")
    pretrained_eval = full_eval(model, "pretrained")
    del model
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [1.4, 1]})
    a1.plot(*zip(*pre_curve), color=PAL["blue"], lw=2); a1.axhline(refs["ntp_floor_pos1plus"], color=PAL["red"], ls="--", lw=1.2)
    a1.text(pre_curve[-1][0], refs["ntp_floor_pos1plus"] + 0.02, "Bayes floor", color=PAL["red"], ha="right", fontsize=8)
    a1.axhline(UNIFORM, color=PAL["muted"], ls=":", lw=1); a1.text(pre_curve[-1][0], UNIFORM - 0.06, "uniform (ln 8)", color=PAL["muted"], ha="right", fontsize=8)
    a1.set_xlabel("pretraining step"); a1.set_ylabel("val NTP loss (nats)"); a1.set_title(f"Pretraining to plateau, m={M_DEMO}", loc="left")
    plot_basis(a2, [pretrained_eval], refs["ntp_floor_per_level"], UNIFORM, ["what the pretrained model knows, per level"])
    plt.tight_layout(); plt.show()

### The full-scale pretrained bases

Same 2.7M-parameter model at every $m$, trained to plateau. Bar length = share of each level's learnable structure actually learned, $(\ln 8 - \text{loss}_\ell)/(\ln 8 - \text{floor}_\ell)$; hatched = nothing to predict at that level (the Bayes floor is already at chance, because root-crossing positions mix); red dashed = the basis boundary.

| m | plateau steps | val (nats) | captured excess entropy |
|---|---|---|---|
| 1 | 3000 | 0.0234 | 0.999 |
| 2 | 7000 | 0.7021 | 0.996 |
| 3 | 13250 | 1.1192 | 0.979 |
| 4 | 14500 | 1.4335 | 0.663 |
| 6 | 7250 | 1.8575 | 0.228 |

Pretraining coverage falls with $m$ as $m^L$ predicts. At $m=4$ and $m=6$ the deep levels (L3+) are essentially unlearned — per-level excess ≈ +0.5 to +1.0 nats, i.e. at uniform. **This is the basis whose boundary the attainment ladder then probes.**

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(14, 3.4), sharey=True)
for ax, m in zip(axes, MS_FULL):
    r = full_refs(m); p = full_pretrain(m)
    plot_basis(ax, [p["eval"]], r["ntp_floor_per_level"], r["uniform_nats"], [f"m={m}  (learned {p['captured']:.3f}, {p['steps'] / 1000:.1f}K steps)"])
fig.suptitle("Full scale: pretraining builds a basis with a boundary — and it sinks as m grows", x=0.01, ha="left", fontsize=11)
plt.tight_layout(); plt.show()

## 6. Part 2 — Attainment: the ladder

All three mechanisms start from the *same* pretrained checkpoint and get the *same* rollout budget (full scale: 384K rollouts = 6000 steps × batch 64), on both verifiers, at every $m$:

1. **Vanilla REINFORCE** — EMA baseline (decay 0.95), no anchor. Plus, in the full run, a from-scratch variant and a step-matched continued-NTP control (`pretrain_only`).
2. **KL-anchored REINFORCE** — `kl_coef = 0.1` to the frozen pretrained policy: the RLHF-like stabilization.
3. **Expert iteration** — rounds of best-of-N selection followed by SFT on the winners (suffix-masked loss): the STaR / rejection-sampling-finetuning recipe. This is the *noiseless, perfectly-credit-assigned limit* of the sample-reweighting class that every policy-gradient method approximates ("increase the probability of your own high-reward samples"). Scope caveat: it idealizes the positive-gradient half of the class; exotic exploration or negative-gradient schemes sit outside the bound.

**Why end at expert iteration.** The "that's just your RL variant" objection can't be answered variant-by-variant. If the *class limit* fails to move the basis boundary, the objection collapses from "your variant is weak" to "the class limit is weak". If it succeeds, capability creation has been localized in the filtered-imitation channel (SFT on verifier-selected samples = pretraining on self-generated grammar data) rather than gradient reweighting. Either outcome is informative.

In [ ]:
# ----------------------------------------------------------------------------
# The three mechanisms (+ the continued-NTP control), ported from rl_dim_ablation.py
# ----------------------------------------------------------------------------
def run_reinforce(start_state, rw, use_kl=False, kl_coef=0.0, label="REINFORCE"):
    model = make_gpt(); model.load_state_dict(start_state)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    ref = None
    if use_kl:
        ref = make_gpt(); ref.load_state_dict(start_state); ref.eval()
        for p in ref.parameters(): p.requires_grad = False
    baseline, traj, ckpts, t0 = None, [], {}, time.time()
    for st in range(RL_STEPS):
        model.eval(); prefix, true = rl_batch()
        with torch.no_grad(): gen = generate_suffix(model, prefix, SUFFIX_LEN)
        reward = reward_fn(prefix, gen, true, rw, rules_t)
        baseline = reward.mean().item() if baseline is None else 0.95 * baseline + 0.05 * reward.mean().item()
        adv = (reward - baseline).detach()
        model.train(); scored = torch.cat([prefix, gen], 1)
        logits, _ = model(scored)
        logp = F.log_softmax(logits[:, PREFIX_LEN - 1:PREFIX_LEN - 1 + SUFFIX_LEN], -1)
        loss = -(adv.unsqueeze(1) * logp.gather(2, gen.unsqueeze(-1)).squeeze(-1)).mean()
        if use_kl:
            with torch.no_grad(): ref_logp = F.log_softmax(ref(scored)[0][:, PREFIX_LEN - 1:PREFIX_LEN - 1 + SUFFIX_LEN], -1)
            loss = loss + kl_coef * (logp.exp() * (logp - ref_logp)).sum(-1).mean()      # KL(pi_theta || pi_ref)
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        if (st + 1) % 25 == 0 or st == 0:
            traj.append({"step": st + 1, "reward": float(reward.mean()), "val_loss_quick": eval_val_loss(model, n_batches=4)})
        if (st + 1) % 200 == 0: print(f"    [{label}] step {st + 1:5d}: reward={reward.mean():.4f} (ema {baseline:.4f}) val~{traj[-1]['val_loss_quick']:.3f}  [{time.time() - t0:.0f}s]")
        if (st + 1) % CKPT_EVERY == 0: ckpts[st + 1] = full_eval(model, f"{label} s{st + 1}")
    del model, opt, ref
    return {"trajectory": traj, "checkpoints": ckpts}

def run_continued_ntp(start_state, label="continued NTP"):
    model = make_gpt(); model.load_state_dict(start_state)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01); ckpts = {}
    for st in range(RL_STEPS):
        model.train(); x, y = ntp_batch(); _, loss = model(x, y)
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        if (st + 1) % CKPT_EVERY == 0: ckpts[st + 1] = full_eval(model, f"{label} s{st + 1}")
    del model, opt
    return {"trajectory": [], "checkpoints": ckpts}

def run_expert_iteration(start_state, rw, label="expert iteration"):
    model = make_gpt(); model.load_state_dict(start_state)
    prompts_per_pass = max(1, 1024 // EI_N); traj, ckpts, t0 = [], {}, time.time()
    for rnd in range(EI_ROUNDS):
        model.eval(); win_seqs, win_r, ssum, scnt, remaining = [], [], 0.0, 0, EI_PROMPTS
        while remaining > 0:                                            # --- best-of-N selection ---
            k = min(prompts_per_pass, remaining); remaining -= k
            seqs = rl_t[torch.randint(N_TRAIN, (k,), generator=ei_gen)].to(device)
            prefix, true = seqs[:, :PREFIX_LEN].repeat_interleave(EI_N, 0), seqs[:, PREFIX_LEN:].repeat_interleave(EI_N, 0)
            with torch.no_grad(): gen = generate_suffix(model, prefix, SUFFIX_LEN)
            r = reward_fn(prefix, gen, true, rw, rules_t).view(k, EI_N); ssum += float(r.sum()); scnt += k * EI_N
            best = r.argmax(1); ar = torch.arange(k, device=device)
            win_seqs.append(torch.cat([seqs[:, :PREFIX_LEN], gen.view(k, EI_N, SUFFIX_LEN)[ar, best]], 1).cpu()); win_r.append(r[ar, best].cpu())
        win = torch.cat(win_seqs); win_r = torch.cat(win_r)
        win_root = float(parse_valid_fractions_torch(win.to(device), rules_t)[:, -1].mean())
        model.train(); opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)   # --- SFT on winners, suffix loss only ---
        for _ in range(EI_SFT_EPOCHS):
            perm = torch.randperm(len(win), generator=ei_gen)
            bsz = min(BATCH, len(win))
            for i in range(0, len(win) - bsz + 1, bsz):
                b = win[perm[i:i + bsz]].to(device); x, y = b[:, :-1].contiguous(), b[:, 1:].contiguous()
                ce = F.cross_entropy(model(x)[0].reshape(-1, V), y.reshape(-1), reduction="none").view(-1, SEQ_LEN - 1)
                loss = ce[:, PREFIX_LEN - 1:].mean()
                opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        del opt
        sel = {"round": rnd + 1, "sample_reward_mean": ssum / scnt, "winner_reward_mean": float(win_r.mean()), "winner_root_valid": win_root}
        traj.append(sel)
        print(f"    [{label}] round {rnd + 1}: sampled reward={sel['sample_reward_mean']:.4f}  winners={sel['winner_reward_mean']:.4f} "
              f"(winners root-valid {win_root:.3f})  [{time.time() - t0:.0f}s]")
        ck = full_eval(model, f"{label} r{rnd + 1}"); ck["selection"] = sel; ckpts[rnd + 1] = ck
    del model
    return {"trajectory": traj, "checkpoints": ckpts}

LIVE = {}
def last_ckpt(res): return res["checkpoints"][max(res["checkpoints"])]

### 6a. Vanilla REINFORCE (live)

Watch three things: the training reward, the quick val loss (which should head *up* — away from the Bayes floor — if the basis is being destroyed), and after training, the greedy-vs-sampled gap (entropy collapse shows as greedy ≡ sampled).

In [ ]:
if RUN_LIVE:
    print(f"REINFORCE, {VERIFIER} verifier, from the pretrained checkpoint")
    LIVE["REINFORCE"] = run_reinforce(pretrained_state, VERIFIER, label="REINFORCE")

In [ ]:
def plot_rl_trajectory(res, label, color):
    tr = res["trajectory"]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.3))
    a1.plot([t["step"] for t in tr], [t["reward"] for t in tr], color=color, lw=1.8)
    a1.set_xlabel("RL step"); a1.set_ylabel(f"training reward ({VERIFIER})"); a1.set_title(f"{label}: reward", loc="left")
    a2.plot([t["step"] for t in tr], [t["val_loss_quick"] for t in tr], color=color, lw=1.8)
    a2.axhline(refs["ntp_floor_pos1plus"], color=PAL["red"], ls="--", lw=1.2); a2.axhline(UNIFORM, color=PAL["muted"], ls=":", lw=1)
    a2.text(tr[-1]["step"], refs["ntp_floor_pos1plus"], " Bayes floor", color=PAL["red"], fontsize=8, va="bottom", ha="right")
    a2.text(tr[-1]["step"], UNIFORM, " uniform", color=PAL["muted"], fontsize=8, va="bottom", ha="right")
    a2.set_xlabel("RL step"); a2.set_ylabel("val NTP loss (nats)"); a2.set_title(f"{label}: what happens to the pretrained distribution", loc="left")
    plt.tight_layout(); plt.show()

if RUN_LIVE:
    plot_rl_trajectory(LIVE["REINFORCE"], "REINFORCE", MECH_COLOR["REINFORCE"])

### What happened at full scale (2a): vanilla REINFORCE self-destructs at every $m$

Pure REINFORCE (no anchor) drove NTP excess to **9–14 nats above the floor at every $m$** — far worse than uniform guessing (2.08 nats) — collapsed policy entropy (greedy ≡ sampled everywhere), and flattened per-layer $\eta^2$ ($m=2$ exact arm: L5 feature $\eta^2$ 0.647 → 0.137 at block 0).

Most strikingly, at $m=1$ — where the pretrained policy sat essentially at the Bayes ceiling (greedy exact 0.849 vs ceiling 0.842) — REINFORCE dragged it to **0.388** while its own training reward fell 0.63 → 0.38. On the parse verifier at $m=2/3$ the reward climbed to ~0.92–0.94 and *then crashed* ($m=2$: 0.92 → 0.42 → 0.55) — **optimizer self-destruction, not signal absence**. The unanchored $m=6$ parse "success" (sampled parse 1.000) was total mode collapse (14.2 nats excess).

Scratch RL never approached pretraining anywhere: even at $m=1$, 384K rollouts left it at 0.378 exact — *below the prefix-blind floor* (0.746) that NTP passes in a few hundred steps.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [1, 1.2]})
x = np.arange(len(MS_FULL))
for j, rw in enumerate(["exact", "parse"]):
    vals = [full_final(m, f"pretrain_rl_{rw}")["val_loss"] - full_refs(m)["ntp_floor_pos1plus"] for m in MS_FULL]
    a1.bar(x + (j - 0.5) * 0.36, vals, width=0.34, color=VERIFIER_COLOR[rw], label=f"{rw} verifier")
    for xi, v_ in zip(x + (j - 0.5) * 0.36, vals): a1.text(xi, v_ + 0.2, f"{v_:.1f}", ha="center", fontsize=8, color=PAL["ink"])
a1.axhline(np.log(8), color=PAL["red"], ls="--", lw=1.2); a1.text(-0.4, np.log(8) + 0.25, "uniform guessing (ln 8)", color=PAL["red"], fontsize=8)
a1.set_xticks(x); a1.set_xticklabels([f"m={m}" for m in MS_FULL]); a1.set_ylabel("NTP loss above the Bayes floor (nats)")
a1.set_title("Full scale: vanilla REINFORCE, final checkpoint", loc="left"); a1.legend(loc="upper left", fontsize=8, bbox_to_anchor=(0, 0.92))
for m, c in [(2, LEVEL_RAMP[2]), (3, LEVEL_RAMP[4])]:
    tr = FULL[str(m)]["conditions"]["pretrain_rl_parse"]["trajectory"]
    a2.plot([t["step"] for t in tr], [t["reward"] for t in tr], color=c, lw=1.8, label=f"m={m}")
a2.set_xlabel("RL step"); a2.set_ylabel("training reward (parse)"); a2.set_title("Full scale: parse-verifier reward climbs, then crashes", loc="left"); a2.legend(fontsize=8)
plt.tight_layout(); plt.show()

### 6b. KL-anchored REINFORCE (live)

In [ ]:
if RUN_LIVE:
    print(f"REINFORCE + KL(coef={KL_COEF}) to the frozen pretrained policy, {VERIFIER} verifier")
    LIVE["REINFORCE + KL"] = run_reinforce(pretrained_state, VERIFIER, use_kl=True, kl_coef=KL_COEF, label="REINFORCE + KL")
    plot_rl_trajectory(LIVE["REINFORCE + KL"], "REINFORCE + KL", MECH_COLOR["REINFORCE + KL"])

### What happened at full scale (2b): the KL anchor cures the destruction and reveals how little there was to gain

With `kl_coef = 0.1`, NTP excess drops from 10–14 to **0.05–0.8 nats** at $m \ge 2$ and entropy collapse disappears. What is left (sampled-policy accuracy, final checkpoint):

| m | exact: pretrained → RL | calibrated ref. (ΣP²) | parse: pretrained → RL | NTP excess (nats) |
|---|---|---|---|---|
| 1 | 0.836 → 0.841 | 0.842 (ceiling) | 0.977 → **0.758** | 2.93 |
| 2 | 0.191 → 0.190 | 0.191 | 0.980 → 0.993 | 0.29 |
| 3 | 0.157 → 0.157 | 0.160 | 0.919 → 0.925 | 0.07 |
| 4 | 0.137 → 0.136 | 0.136 | 0.817 → 0.834 | 0.42 |
| 6 | 0.130 → 0.133 | 0.130 | 0.731 → 0.743 | 0.79 |

- **Exact verifier**: zero gain at every $m \ge 2$ — sampled accuracy sits at the calibrated $\sum P^2$ reference (0.190 vs 0.191 at $m=2$, stable across checkpoints), greedy where pretraining left it. At $m=1$ the anchor fixes the pathology: anchored RL *holds* the ceiling (0.846) instead of destroying it. Exactly as bound B1 requires.
- **Parse verifier**: small genuine polish at $m \ge 2$ (sampled parse +0.006 to +0.017; root validity 0.902 → 0.961 at $m=2$) at small NTP cost. At $m=1$ it still harmed (0.977 → 0.758, root 0.930 → 0.243, 2.9 nats excess) — a variant artifact, as the class-limit arm then showed.

Stabilizing RL turns destruction into **stasis** — not into learning.

### 6c. Expert iteration — the class limit (live)

Each round: sample N continuations per prompt, keep the reward-argmax, SFT on the winners (loss on suffix positions only), repeat. Rollout-matched to the REINFORCE runs. Watch **root-level validity of the sampled policy** (the deep-structure metric) and, side by side, **per-level NTP excess at the levels pretraining never learned**.

In [ ]:
if RUN_LIVE:
    print(f"Expert iteration, {VERIFIER} verifier: {EI_ROUNDS} rounds x {EI_PROMPTS} prompts x best-of-{EI_N}")
    LIVE["expert iteration"] = run_expert_iteration(pretrained_state, VERIFIER)

In [ ]:
def plot_dissociation(rounds, cks, floors, uniform, pretrained_ck=None, title="", levels=(3, 4)):
    """Top: root validity of the sampled policy by round. Bottom: NTP excess at the given (unlearned) levels, with chance lines."""
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(7.5, 5.6), sharex=True, gridspec_kw={"height_ratios": [1, 1.1]})
    root = [ck["generation"]["sampled"]["parse_per_level"][-1] for ck in cks]
    xs = list(rounds)
    if pretrained_ck is not None:
        xs = [0] + xs; root = [pretrained_ck["generation"]["sampled"]["parse_per_level"][-1]] + root; cks = [pretrained_ck] + list(cks)
    a1.plot(xs, root, color=PAL["orange"], lw=2, marker="o", ms=6)
    a1.text(xs[-1], root[-1], f"  {root[-1]:.3f}", va="center", fontsize=9, color=PAL["ink"]); a1.text(xs[0], root[0], f"{root[0]:.3f}  ", va="center", ha="right", fontsize=9, color=PAL["ink"])
    a1.set_ylabel("root-level validity\n(sampled policy)"); a1.set_title(title, loc="left")
    for l in levels:
        ex = [ck["per_level_ntp"][f"L{l}"] - floors[l] for ck in cks]
        a2.plot(xs, ex, color=LEVEL_RAMP[l], lw=2, marker="o", ms=6, label=f"L{l} excess")
        a2.axhline(uniform - floors[l], color=LEVEL_RAMP[l], ls="--", lw=1); a2.text(xs[-1], uniform - floors[l], f"  chance, L{l}", va="center", fontsize=8, color=LEVEL_RAMP[l])
    a2.set_ylabel("NTP loss above the Bayes floor\nat the unlearned levels (nats)"); a2.set_xlabel("expert-iteration round (0 = pretrained)"); a2.legend(fontsize=8, loc="lower left")
    a2.set_xticks(xs); plt.tight_layout(); plt.show()

if RUN_LIVE:
    ei = LIVE["expert iteration"]
    cap = captured_per_level(pretrained_eval, refs["ntp_floor_per_level"], UNIFORM)
    unlearned = [l for l in range(DEPTH) if cap[l] is not None and cap[l] < 0.5][:2] or [DEPTH - 3, DEPTH - 2]
    plot_dissociation(sorted(ei["checkpoints"]), [ei["checkpoints"][r] for r in sorted(ei["checkpoints"])], refs["ntp_floor_per_level"], UNIFORM,
                      pretrained_ck=pretrained_eval, title=f"Live replica, m={M_DEMO}: pass rate vs. knowledge of the unlearned levels", levels=unlearned)

### What happened at full scale (2c): expert iteration is far stronger — and still never moves the basis boundary

Sampled-policy parse validity (the distribution, not just the argmax) and root-level validity, final round:

| m | pretrained | REINFORCE | +KL | **EI** | root: pretrained | +KL | **EI** | NTP excess: KL | **EI** |
|---|---|---|---|---|---|---|---|---|---|
| 1 | 0.977 | 0.500 | 0.758 | **1.000** | 0.930 | 0.243 | **0.999** | 2.93 | 0.28 |
| 2 | 0.980 | 0.552 | 0.993 | 0.981 | 0.902 | 0.961 | 0.922 | 0.29 | 0.11 |
| 3 | 0.919 | 0.583 | 0.925 | **0.932** | 0.556 | 0.570 | **0.644** | 0.08 | 0.20 |
| 4 | 0.817 | 0.901\* | 0.834 | **0.884** | 0.248 | 0.288 | **0.447** | 0.42 | 0.65 |
| 6 | 0.731 | 1.000\* | 0.743 | **0.800** | 0.153 | 0.160 | **0.292** | 0.79 | 0.89 |

\*mode-collapsed (10–14 nats NTP excess). EI's values are honest distributional improvements — the STaR-style flywheel visibly spins (winner data quality at $m=6$ rises 89% → 99% root-valid round over round), no entropy collapse, $m=1$ fully recovered (so the anchored harm at $m=1$ was the *variant*, not the class). EI nearly **doubles** root validity at $m=4$ (0.25 → 0.45) and $m=6$ (0.15 → 0.29). Whatever this class can do, this arm does more of it than policy gradients.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
x = np.arange(len(MS_FULL)); w = 0.26
series = [("pretrained", lambda m: full_pretrain(m)["eval"]), ("REINFORCE + KL", lambda m: full_final(m, "pretrain_rl_kl_parse")), ("expert iteration", lambda m: full_final(m, "ei_parse"))]
for j, (name, get) in enumerate(series):
    vals = [get(m)["generation"]["sampled"]["parse_per_level"][-1] for m in MS_FULL]
    ax.bar(x + (j - 1) * w, vals, width=w - 0.02, color=MECH_COLOR[name], label=name)
    for xi, v_ in zip(x + (j - 1) * w, vals): ax.text(xi, v_ + 0.015, f"{v_:.2f}", ha="center", fontsize=7.5, color=PAL["ink"])
ax.set_xticks(x); ax.set_xticklabels([f"m={m}" for m in MS_FULL]); ax.set_ylim(0, 1.08); ax.set_ylabel("root-level validity (sampled policy)")
ax.set_title("Full scale, parse verifier: share of sampled generations that parse all the way to the root, final checkpoint", loc="left"); ax.legend(fontsize=8, loc="upper right")
plt.tight_layout(); plt.show()

### The headline result: the dissociation

At $m=4$ and $m=6$, round by round, **root validity climbs monotonically while knowledge of the deep rules — per-level NTP excess at the levels pretraining never learned — stays pinned at uniform and *slightly worsens*:**

| round ($m=6$) | root validity (sampled) | L3 excess (nats) | L4 excess (nats) |
|---|---|---|---|
| 1 | 0.142 | +0.969 | +1.027 |
| 2 | 0.174 | +0.974 | +1.027 |
| 3 | 0.233 | +0.983 | +1.032 |
| 4 | 0.232 | +0.978 | +1.028 |
| 5 | 0.283 | +0.982 | +1.030 |
| 6 | 0.292 | +0.991 | +1.039 |

($m=4$ shows the same pattern: root 0.280 → 0.447 while L3 excess +0.511 → +0.564.) Per-layer $\eta^2$ shows no new hierarchical structure anywhere ($m=6$ block 0 L5: 0.104 pretrained → 0.109 after EI).

The class-limit mechanism **doubled performance on a deep-structure-requiring metric without learning any deep structure** — probability-mass redistribution over trajectories the basis could already produce, selected by the verifier. In language terms: *pass rates up, the model still doesn't know the rules.* This dissociation is only measurable because the data-generating process is known.

In [ ]:
for m in (6, 4):
    r = full_refs(m); rounds = full_rounds(m, "ei_parse")
    plot_dissociation(range(1, len(rounds) + 1), rounds, r["ntp_floor_per_level"], r["uniform_nats"], pretrained_ck=full_pretrain(m)["eval"],
                      title=f"Full scale, m={m}, expert iteration: pass rate doubles, the model learns no new rule", levels=(3, 4))

In [ ]:
# The basis before and after, per mechanism (full scale, parse verifier): no band lights up
for m in (6, 4):
    r = full_refs(m)
    cks = [full_pretrain(m)["eval"], full_final(m, "pretrain_rl_parse"), full_final(m, "pretrain_rl_kl_parse"), full_final(m, "ei_parse"), full_final(m, "pretrain_only")]
    fig, axes = plt.subplots(1, 5, figsize=(14, 3.2), sharey=True)
    plot_basis(axes, cks, r["ntp_floor_per_level"], r["uniform_nats"],
               ["pretrained", "REINFORCE (collapsed)", "REINFORCE + KL", "expert iteration", "continued NTP (control)"],
               suptitle=f"Full scale, m={m}: what the model knows per level, after each mechanism (parse verifier)")
    plt.tight_layout(); plt.show()

### The price steepens with $m$

**The exact-verifier $m$-gradient, at the class limit.** EI-exact's per-rollout improvement falls monotonically with $m$ — sampled reward over 6 rounds: +0.020 ($m=2$), +0.011 ($m=3$), +0.002 ($m=4$), ~+0.002 ($m=6$) — the empirical attainment curve matching bound B1's headroom collapse. The gains that do occur are anchor-free *sharpening* ($m=2$: sampled 0.191 → 0.215, above the calibrated $\sum P^2$ reference and toward the greedy ceiling, with only 0.17 nats excess).

**Representational cost per unit reward grows with $m$.** Within EI on the parse verifier, NTP excess rises 0.11 → 0.20 → 0.65 → 0.89 nats across $m = 2 → 6$ even as reward gains grow — at higher semantic dimensionality the same mechanism trades away more of the pretrained distribution per unit of reward. Expressed as the share of pretrained knowledge erased (1 − captured-after / captured-before): **7% at $m=2$ → 61% at $m=6$**. The $m=1$ EI model is the miniature: parse validity 1.000 with the distribution among the 8 legal sequences miscalibrated (L3 excess +0.706) — perfectly grammatical, distributionally distorted.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.6))
x = np.arange(len(MS_FULL))
gain = [FULL[str(m)]["conditions"]["ei_exact"]["trajectory"][-1]["sample_reward_mean"] - FULL[str(m)]["conditions"]["ei_exact"]["trajectory"][0]["sample_reward_mean"] for m in MS_FULL]
a1.bar(x, gain, width=0.6, color=VERIFIER_COLOR["exact"])
for xi, g in zip(x, gain): a1.text(xi, g + 0.0006, f"+{g:.3f}", ha="center", fontsize=9, color=PAL["ink"])
a1.set_xticks(x); a1.set_xticklabels([f"m={m}" for m in MS_FULL]); a1.set_ylabel("sampled exact-match reward gain, rounds 1→6")
a1.set_title("Canonical verifier: EI's gain → 0 with m, tracking the headroom bound", loc="left"); a1.text(0, 0.0002, "already at\nthe ceiling", ha="center", fontsize=7.5, color="white")
erased = []
for m in MS_FULL:
    r = full_refs(m); span = r["uniform_nats"] - r["ntp_floor_pos1plus"]
    before = (r["uniform_nats"] - full_pretrain(m)["best_val"]) / span; after = (r["uniform_nats"] - full_final(m, "ei_parse")["val_loss"]) / span
    erased.append(1 - after / before)
a2.bar(x, erased, width=0.6, color=VERIFIER_COLOR["parse"])
for xi, e in zip(x, erased): a2.text(xi, e + 0.01, f"{e * 100:.0f}%", ha="center", fontsize=9, color=PAL["ink"])
a2.set_xticks(x); a2.set_xticklabels([f"m={m}" for m in MS_FULL]); a2.set_ylabel("share of pretrained knowledge erased by EI")
a2.set_title("Validity verifier: the same six rounds erase more of the basis as m grows", loc="left"); a2.set_ylim(0, 0.75)
plt.tight_layout(); plt.show()

### Live replica: everything side by side

The continued-NTP control is what "just keep pretraining" for the same number of gradient steps does to the same readouts.

In [ ]:
if RUN_LIVE:
    LIVE["continued NTP"] = run_continued_ntp(pretrained_state)
    names = ["pretrained", "REINFORCE", "REINFORCE + KL", "expert iteration", "continued NTP"]
    cks = [pretrained_eval] + [last_ckpt(LIVE[n]) for n in names[1:]]
    fig, axes = plt.subplots(1, 5, figsize=(14, 3.2), sharey=True)
    plot_basis(axes, cks, refs["ntp_floor_per_level"], UNIFORM, names, suptitle=f"Live replica, m={M_DEMO}, {VERIFIER} verifier: what the model knows per level, after each mechanism")
    plt.tight_layout(); plt.show()

    fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(14, 3.6))
    x = np.arange(len(names)); cols = [MECH_COLOR[n] for n in names]
    ex = [ck["val_loss"] - refs["ntp_floor_pos1plus"] for ck in cks]
    a1.bar(x, ex, color=cols, width=0.6); a1.axhline(UNIFORM - refs["ntp_floor_pos1plus"], color=PAL["red"], ls="--", lw=1.2)
    a1.text(len(names) - 0.6, UNIFORM - refs["ntp_floor_pos1plus"], "uniform ", color=PAL["red"], fontsize=8, ha="right", va="bottom")
    a1.set_ylabel("NTP loss above Bayes floor (nats)"); a1.set_title("Distributional damage", loc="left")
    for j, mode in enumerate(("greedy", "sampled")):
        a2.bar(x + (j - 0.5) * 0.3, [ck["generation"][mode]["parse_per_level"][-1] for ck in cks], width=0.28, color=cols, alpha=1 if j else 0.45, label=mode)
    a2.set_ylabel("root-level validity"); a2.set_title("Task metric: root validity (light = greedy, solid = sampled)", loc="left", fontsize=9.5); a2.set_ylim(0, 1.05)
    for j, ck in enumerate(cks):
        a3.plot(range(DEPTH), ck["feature_eta2"]["post_block0"], color=cols[j], lw=1.8, marker="o", ms=4, label=names[j])
    a3.set_xticks(range(DEPTH)); a3.set_xticklabels([f"L{l}" for l in range(DEPTH)]); a3.set_ylabel("feature η² (block 0, last position)"); a3.set_title("Hierarchy structure in the activations", loc="left"); a3.legend(fontsize=7.5)
    for ax in (a1, a2): ax.set_xticks(x); ax.set_xticklabels(names, rotation=20, ha="right", fontsize=8)
    plt.tight_layout(); plt.show()

    print(f"{'condition':>18} | {'NTP excess':>10} | {'greedy parse':>12} | {'sampled parse':>13} | {'root (sampled)':>14} | {'greedy exact':>12} | {'sampled exact':>13} | " + " ".join(f"{'L' + str(l) + ' exc':>7}" for l in range(DEPTH)))
    for n, ck in zip(names, cks):
        g, s_ = ck["generation"]["greedy"], ck["generation"]["sampled"]
        print(f"{n:>18} | {ck['val_loss'] - refs['ntp_floor_pos1plus']:>+10.3f} | {g['parse']:>12.3f} | {s_['parse']:>13.3f} | {s_['parse_per_level'][-1]:>14.3f} | {g['exact']:>12.3f} | {s_['exact']:>13.3f} | "
              + " ".join(f"{e:>+7.3f}" for e in excess_per_level(ck, refs["ntp_floor_per_level"])))

## 7. Findings

1. **[Bound] Canonical-answer verifiers lose their semantic content as synonymy grows** — prefix-conditional exact-match headroom 0.096 → ~0 across $m = 1 → 6$ by exact computation; validity verifiers gain reward density on the same knob. "RL is bad on complex domains" is, in this substrate, a theorem about the verifier × domain pair before it is a fact about any algorithm.

2. **[Attainment] Across the full ladder — vanilla REINFORCE, KL-anchored REINFORCE, and the expert-iteration class limit, × 5 dimensionalities × 2 verifiers — reward optimization never created structure the pretrained basis lacked.** Deep-level rule knowledge (per-level NTP excess) and $\eta^2$ hierarchy structure stayed at pretrained values in every condition; the capability boundary sat exactly where pretraining left it. What varied by mechanism was everything else: vanilla PG *destroyed* the basis (9–14 nats), the KL anchor *preserved* it with near-zero gains, and EI produced large, honest task gains by *selection and sharpening within it*.

3. **[Attainment] The reward/representation trade steepens with dimensionality**: EI's NTP damage per unit of parse-reward gain grows with $m$, and EI-exact's climb rate falls to zero with $m$. The "models get more fried as tasks get more sophisticated" pattern appears in the best-behaved member of the class, not just the fragile ones.

4. **[Method] The class-limit arm is what makes 2–3 robust to the "your RL variant" objection**: EI is strictly stronger than the policy-gradient variants here (it alone recovers $m=1$), so the immobile basis boundary cannot be attributed to a weak optimizer — while the honest scope note stands (exploration-bonus / negative-gradient schemes are outside the idealization).

## 8. What it means

**Reward optimization is selection within the basis.** Picture the set of trajectories the pretrained model can produce as a region; the verifier marks some of them as passing. Reward optimization moves probability mass toward the passing ones. The outline of the region is the same before and after — only the mass moved. A trajectory that needs a rule the basis lacks is unreachable either way. Across 3 mechanisms × 5 $m$ × 2 verifiers, deep-rule knowledge stayed where pretraining left it; REINFORCE *burned* the region, the anchor *froze* it, EI gained by selection and sharpening within it. In language terms: pass rates up, rules still unknown, the distribution paid — and only a known generating process can tell these apart.

**Not all language tasks are created equal.** Code and math have few right answers (low $m$); creative writing and original thinking are inherently synonymous (high $m$). Unsurprisingly, language-model RL works on the former and not on the latter. The one channel that *could* teach a missing rule is the pretraining-shaped one: SFT on selected samples is more data. At the RL-matched budget it did not, while reward doubled.

**Takeaway.** Continual learning that *expands* what a model can do needs a mechanism that moves the basis boundary. Vanilla policy reward is, structurally, not it.

### Scope

Interpretation is deliberately scoped: this is one substrate, one model size, single seed (effects are large, monotone in $m$, and internally replicated across three mechanisms and two verifiers, which triangulates seed sensitivity about as well as one seed can), and EI's validity curves were still climbing at round 6 — though its deep-level excess trend across rounds is flat-to-worsening, which is the falsifiable part. Where capability creation *would* show up in this framework is the filtered-imitation channel scaled further: SFT on verifier-selected samples is pretraining on self-generated grammar data, and nothing here rules out that channel teaching missing rules with far more data — the observation is that at the RL-matched budget it demonstrably did not, while task reward doubled anyway.

The live replica in this notebook is scaled down (~1/5 of the rollout budget, shorter pretraining); its numbers will be noisier and its effects smaller than the tables above, but the *readouts* — per-level excess vs the Bayes floor, root validity, $\eta^2$ — are the same code as the full run.

## 9. Reproducing the full-scale runs

The full experiment lives at [`experiments/rhm/rl_dimensionality/`](https://github.com/jagilley/abstraction/tree/main/experiments/rhm/rl_dimensionality) (`rl_dim_ablation.py`; design record in `DESIGN.md`, writeup in `README.md`, provenance in `CONVERSATION.md`) and runs on Modal:

```bash
cd experiments/
modal run -m rhm.rl_dimensionality.rl_dim_ablation::self_test            # torch parse == numpy parse; BP == brute force
for M in 1 2 3 4 6; do                                                    # base sweep: pretrain-to-plateau + scratch_rl / pretrain_rl / pretrain_only
  modal run --detach -m rhm.rl_dimensionality.rl_dim_ablation::run_setting --m $M
done
for M in 1 2 3 4 6; do                                                    # KL-anchored arm
  modal run --detach -m rhm.rl_dimensionality.rl_dim_ablation::run_setting --m $M \
    --only-kl --kl-coef 0.1 --run-tag kl01 --pretrained-path rl_dimensionality/v8_s2_L6_m${M}_both_seed42/pretrained.pt
done
for M in 1 2 3 4 6; do                                                    # expert-iteration arm (rollout-matched: 6 x 4000 x 16 = 384K)
  modal run --detach -m rhm.rl_dimensionality.rl_dim_ablation::run_setting --m $M \
    --only-ei --run-tag ei01 --pretrained-path rl_dimensionality/v8_s2_L6_m${M}_both_seed42/pretrained.pt
done
```

The embedded `FULL` dictionary in this notebook is a compact extract of those 15 `results.json` files (final and per-round checkpoints, reward trajectories, per-level NTP, generation metrics, per-layer $\eta^2$).

*Random Hierarchy Model: Cagnetta, Petrini, Tomasini, Favero & Wyart, "How Deep Neural Networks Learn Compositional Data: The Random Hierarchy Model", Phys. Rev. X 14, 031001 (2024).*